<a href="https://colab.research.google.com/github/shaan-byte/python_ds_colab/blob/main/Introduction_to_Pandas_DataFrames.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 6.1 — Introduction to Pandas DataFrames

**Module 2: Data Analysis Foundations**  
**Week 6 | Session 6.1**

---

## What we will cover today

1. Why Pandas exists — what NumPy cannot do on its own
2. Series — the single-column building block
3. DataFrames — tables with labels
4. Reading a CSV file into a DataFrame
5. Basic inspection — understanding a new dataset quickly

---

> **A new chapter:** For the last three sessions we worked with NumPy — raw arrays of numbers. Today we step into Pandas, which is the tool data scientists actually spend most of their time in. The dataset changes too: we move from cricket scores to an employee records dataset from a mid-sized company. This is much closer to what you will encounter in industry — messy, mixed-type, real-world tabular data with names, dates, departments, and salaries all living together in the same file.

---
## Setup — Run this first

In [ ]:
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)
print("NumPy version: ", np.__version__)
print("Ready.")

Pandas version: 2.2.2
NumPy version:  2.0.2
Ready.


---
# Part 1 — Why Pandas? What NumPy Cannot Do

## The core limitation of NumPy

NumPy arrays are powerful, but they have one fundamental constraint: **every element in an array must be the same type**. An array is all integers, or all floats, or all strings — never a mix.

Real-world data is never like that. A typical row in a company's HR system looks like this:

| employee_id | name | department | salary | join_date | active |
|---|---|---|---|---|---|
| 1001 | Ananya Sharma | Engineering | 95000 | 2021-03-15 | True |

That single row contains an integer, two strings, a float, a date, and a boolean — six different types. A NumPy array cannot hold this without converting everything to strings and losing all numeric meaning.

Pandas was built to solve exactly this problem. It gives you:

| Feature | NumPy array | Pandas DataFrame |
|---|---|---|
| Data types | One type per array | Different types per column |
| Column access | By index number only | By name: `df['salary']` |
| Row access | By index number only | By label or index |
| Missing values | No built-in handling | Native `NaN` support |
| Reading files | Manual parsing | `pd.read_csv()` in one line |
| Built-in stats | Basic aggregations | `.describe()`, `.groupby()`, and much more |

**The relationship:** Pandas is built on top of NumPy. Every column in a Pandas DataFrame is a NumPy array underneath. The speed and vectorization you learned in the last three sessions still applies — Pandas just adds a much more convenient and expressive interface on top.

---
# Part 2 — Series: The Building Block

## What is a Series?

A **Series** is a one-dimensional labelled array. Think of it as a single column from a spreadsheet — values on one side, labels (the index) on the other.

```
Index   Value
  0     95000       <- Ananya's salary
  1     82000       <- Rohit's salary
  2    110000       <- Priya's salary
  3     78000       <- Vikram's salary
Name: salary
```

The index does not have to be numbers — it can be names, dates, or any label that makes sense for your data.

In [ ]:
# Creating a Series from a Python list
# This is the salary column for 6 employees

salaries = pd.Series([95000, 82000, 110000, 78000, 125000, '67000'])

print(salaries)
print()
print("Type:", type(salaries))
print("dtype:", salaries.dtype)   # the data type of the values
print("shape:", salaries.shape)   # just like NumPy

0     95000
1     82000
2    110000
3     78000
4    125000
5     67000
dtype: object

Type: <class 'pandas.core.series.Series'>
dtype: object
shape: (6,)


In [ ]:
salaries.mean()

TypeError: unsupported operand type(s) for +: 'int' and 'str'

In [ ]:
salaries[2]

np.int64(110000)

In [ ]:
# A Series with a meaningful index — employee names instead of 0,1,2...

salaries = pd.Series(
    data  = [95000, 82000, 110000, 78000, 125000, 67000],
    index = ["Ananya", "Rohit", "Priya", "Vikram", "Sneha", "Arjun"],
    name  = "salary"
)

print(salaries)
print()
print("Access by label:  salaries['Priya'] =", salaries["Priya"])
print("Access by position: salaries[2] =",  salaries.iloc[2])

Ananya     95000
Rohit      82000
Priya     110000
Vikram     78000
Sneha     125000
Arjun      67000
Name: salary, dtype: int64

Access by label:  salaries['Priya'] = 110000
Access by position: salaries[2] = 110000


In [ ]:
salaries['Arjun']

np.int64(67000)

In [ ]:
salaries.iloc[3]

np.int64(78000)

In [ ]:
# Series supports the same vectorized operations as NumPy arrays
# — because it IS a NumPy array underneath

print("Mean salary:    ", salaries.mean())
print("Max salary:     ", salaries.max())
print("Min salary:     ", salaries.min())
print()

# Arithmetic — applied to every element
salary_with_raise = salaries * 1.10   # 10% raise for everyone
print("After 10% raise:")
print(salary_with_raise)
print()

# Boolean masking — same as NumPy but labels are preserved
high_earners = salaries[salaries > 90000]
print("Employees earning above 90,000:")
print(high_earners)

Mean salary:     92833.33333333333
Max salary:      125000
Min salary:      67000

After 10% raise:
Ananya    104500.0
Rohit      90200.0
Priya     121000.0
Vikram     85800.0
Sneha     137500.0
Arjun      73700.0
Name: salary, dtype: float64

Employees earning above 90,000:
Ananya     95000
Priya     110000
Sneha     125000
Name: salary, dtype: int64


In [ ]:
# The key difference between a Series and a NumPy array: labels stay with the data
# This matters when you sort or filter — the association is never lost

print("Sorted Series (labels travel with values):")
print(salaries.sort_values())
print()
print("Notice: even after sorting, each value still knows which employee it belongs to.")
print("With a raw NumPy array, sorting the values would break the name-to-salary link.")

Sorted Series (labels travel with values):
Arjun      67000
Vikram     78000
Rohit      82000
Ananya     95000
Priya     110000
Sneha     125000
Name: salary, dtype: int64

Notice: even after sorting, each value still knows which employee it belongs to.
With a raw NumPy array, sorting the values would break the name-to-salary link.


---
### Exercise 1 — Working with a Series

**Task:** Complete the operations on the `experience_years` Series below.

In [ ]:
experience_years = pd.Series(
    data  = [4, 7, 12, 2, 9, 1, 15, 6],
    index = ["Ananya", "Rohit", "Priya", "Vikram", "Sneha", "Arjun", "Deepa", "Kiran"],
    name  = "experience_years"
)

# Q1: How many employees are in this Series?
q1 = len(experience_years)

# Q2: What is the average years of experience?
q2 = experience_years.mean()

# Q3: Which employees have more than 8 years of experience?
#     Return a Series of just those employees
q3 = experience_years[experience_years > 8]

# Q4: Priya's years of experience (access by label)
q4 = experience_years['Priya']

# Q5: Sort the Series from most to least experienced
q5 = experience_years.sort_values()


print(f"Q1 Count:       {q1}")
print(f"Q2 Average:     {q2:.1f}")
print(f"Q3 >8 years:")
print(q3)
print(f"Q4 Priya:       {q4}")
print(f"Q5 Sorted:")
print(q5)
print()
print(f"Correct Q1: {q1 == 8}")
print(f"Correct Q2: {round(q2, 1) == 7.0}")
print(f"Correct Q3: {'Priya' in q3.index and 'Deepa' in q3.index}")
print(f"Correct Q4: {q4 == 12}")

Q1 Count:       8
Q2 Average:     7.0
Q3 >8 years:
Priya    12
Sneha     9
Deepa    15
Name: experience_years, dtype: int64
Q4 Priya:       12
Q5 Sorted:
Arjun      1
Vikram     2
Ananya     4
Kiran      6
Rohit      7
Sneha      9
Priya     12
Deepa     15
Name: experience_years, dtype: int64

Correct Q1: True
Correct Q2: True
Correct Q3: True
Correct Q4: True


<details>
<summary>Show solution</summary>

```python
q1 = len(experience_years)
q2 = experience_years.mean()
q3 = experience_years[experience_years > 8]
q4 = experience_years["Priya"]
q5 = experience_years.sort_values(ascending=False)
```

</details>

---
# Part 3 — DataFrames: Tables with Labels

## What is a DataFrame?

A **DataFrame** is a two-dimensional labelled table. Think of it as a spreadsheet or a database table — rows and columns, where every column has a name and can hold a different data type.

```
         employee_id    name       department   salary    active
    0         1001   Ananya Sharma  Engineering   95000      True
    1         1002   Rohit Mehta    Sales         82000      True
    2         1003   Priya Nair     HR           110000     False
    ...                                                         
    ^                                                           
   row index                                                   
```

A DataFrame is, in essence, a dictionary of Series — each column is one Series, and all columns share the same index (the row labels).

## Whiteboard: Series vs DataFrame

```
Series                           DataFrame

One column only                  Many columns, each a different type
One label axis (the index)       Two label axes (row index + column names)
Like a single spreadsheet column Like a full spreadsheet
pd.Series(data, index=...)       pd.DataFrame({col: values, ...})
```

In [ ]:
# Creating a DataFrame from a dictionary
# Each key is a column name, each value is a list of column values

employees_raw = {
    "employee_id": [1001, 1002, 1003, 1004, 1005, 1006],
    "name":        ["Ananya Sharma", "Rohit Mehta", "Priya Nair",
                    "Vikram Rao",   "Sneha Pillai", "Arjun Das"],
    "department":  ["Engineering", "Sales", "HR", "Finance", "Engineering", "Sales"],
    "salary":      [95000, 82000, 110000, 78000, 125000, 67000],
    "experience":  [4, 7, 12, 2, 9, 1],
    "active":      [True, True, False, True, True, True]
}

df = pd.DataFrame(employees_raw)

print(df)
print()
print("Type:", type(df))

   employee_id           name   department  salary  experience  active
0         1001  Ananya Sharma  Engineering   95000           4    True
1         1002    Rohit Mehta        Sales   82000           7    True
2         1003     Priya Nair           HR  110000          12   False
3         1004     Vikram Rao      Finance   78000           2    True
4         1005   Sneha Pillai  Engineering  125000           9    True
5         1006      Arjun Das        Sales   67000           1    True

Type: <class 'pandas.core.frame.DataFrame'>


In [ ]:
df

,employee_id,name,department,salary,experience,active
0,1001,Ananya Sharma,Engineering,95000,4,True
1,1002,Rohit Mehta,Sales,82000,7,True
2,1003,Priya Nair,HR,110000,12,False
3,1004,Vikram Rao,Finance,78000,2,True
4,1005,Sneha Pillai,Engineering,125000,9,True
5,1006,Arjun Das,Sales,67000,1,True


In [ ]:
mask = pd.Series([True, False, False, True, False, True])

df[mask]

,employee_id,name,department,salary,experience,active
0,1001,Ananya Sharma,Engineering,95000,4,True
3,1004,Vikram Rao,Finance,78000,2,True
5,1006,Arjun Das,Sales,67000,1,True


In [ ]:
df = df.set_index(df['employee_id'])

In [ ]:
df


,employee_id,name,department,salary,experience,active
employee_id,,,,,,
1001,1001,Ananya Sharma,Engineering,95000,4,True
1002,1002,Rohit Mehta,Sales,82000,7,True
1003,1003,Priya Nair,HR,110000,12,False
1004,1004,Vikram Rao,Finance,78000,2,True
1005,1005,Sneha Pillai,Engineering,125000,9,True
1006,1006,Arjun Das,Sales,67000,1,True


In [ ]:
# Each column holds a different type — this is what NumPy alone could not do

print("Column data types:")
print(df.dtypes)
print()
print("Notice: int64, object (string), bool — all in the same structure.")
print("A NumPy array can hold only one type at a time.")

Column data types:
employee_id     int64
name           object
department     object
salary          int64
experience      int64
active           bool
dtype: object

Notice: int64, object (string), bool — all in the same structure.
A NumPy array can hold only one type at a time.


In [ ]:
# Accessing columns
# Method 1: bracket notation — always works, preferred for columns with spaces or special characters
salary_col = df["salary"]
print("df['salary']:")
print(salary_col)
print("Type:", type(salary_col))   # it IS a Series
print()

# Method 2: dot notation — only works for simple column names with no spaces
dept_col = df.department
print("df.department:")
print(dept_col)
print()

# Accessing multiple columns — pass a list of column names
# Note the double brackets: outer selects from df, inner is the list
subset = df[["name", "salary", "department"]]
print("df[['name', 'salary', 'department']]:")
print(subset)

df['salary']:
0     95000
1     82000
2    110000
3     78000
4    125000
5     67000
Name: salary, dtype: int64
Type: <class 'pandas.core.series.Series'>

df.department:
0    Engineering
1          Sales
2             HR
3        Finance
4    Engineering
5          Sales
Name: department, dtype: object

df[['name', 'salary', 'department']]:
            name  salary   department
0  Ananya Sharma   95000  Engineering
1    Rohit Mehta   82000        Sales
2     Priya Nair  110000           HR
3     Vikram Rao   78000      Finance
4   Sneha Pillai  125000  Engineering
5      Arjun Das   67000        Sales


In [ ]:
df[['name', 'salary']]

,name,salary
0,Ananya Sharma,95000
1,Rohit Mehta,82000
2,Priya Nair,110000
3,Vikram Rao,78000
4,Sneha Pillai,125000
5,Arjun Das,67000


In [ ]:
# Accessing rows
# .loc[]  — by LABEL (row label / index value)
# .iloc[] — by INTEGER POSITION (0, 1, 2, ...)

print(".iloc[0] — first row by position:")
print(df.iloc[0])
print()

# print(".iloc[2:4] — rows at positions 2 and 3:")
# print(df.iloc[2:4])
# print()

print(df.loc[1001])
print()

# Accessing a single cell: .loc[row, column]
cell = df.loc[1, "salary"]
print(f".loc[1, 'salary'] — Rohit's salary: {cell}")

.iloc[0] — first row by position:
employee_id             1001
name           Ananya Sharma
department       Engineering
salary                 95000
experience                 4
active                  True
Name: 1001, dtype: object

employee_id             1001
name           Ananya Sharma
department       Engineering
salary                 95000
experience                 4
active                  True
Name: 1001, dtype: object



KeyError: 1

In [ ]:
df

,,employee_id,salary,experience,active,monthly_salary
name,department,,,,,
Ananya Sharma,Engineering,1001,95000,4,True,7916.666667
Rohit Mehta,Sales,1002,82000,7,True,6833.333333
Priya Nair,HR,1003,110000,12,False,9166.666667
Vikram Rao,Finance,1004,78000,2,True,6500.000000
Sneha Pillai,Engineering,1005,125000,9,True,10416.666667
Arjun Das,Sales,1006,67000,1,True,5583.333333


In [ ]:
df = df.set_index(['name', 'department'])
df

,,employee_id,salary,experience,active,monthly_salary
name,department,,,,,
Ananya Sharma,Engineering,1001,95000,4,True,7916.666667
Rohit Mehta,Sales,1002,82000,7,True,6833.333333
Priya Nair,HR,1003,110000,12,False,9166.666667
Vikram Rao,Finance,1004,78000,2,True,6500.000000
Sneha Pillai,Engineering,1005,125000,9,True,10416.666667
Arjun Das,Sales,1006,67000,1,True,5583.333333


In [ ]:
df.loc['Ananya Sharma', 'Engineering']

,Ananya Sharma
,Engineering
employee_id,1001
salary,95000
experience,4
active,True
monthly_salary,7916.666667


In [ ]:
sorted_df = df.sort_values(by='name')

In [ ]:
sorted_df

,employee_id,name,department,salary,experience,active
employee_id,,,,,,
1001,1001,Ananya Sharma,Engineering,95000,4,True
1006,1006,Arjun Das,Sales,67000,1,True
1003,1003,Priya Nair,HR,110000,12,False
1002,1002,Rohit Mehta,Sales,82000,7,True
1005,1005,Sneha Pillai,Engineering,125000,9,True
1004,1004,Vikram Rao,Finance,78000,2,True


In [ ]:
sorted_df.iloc[1]

,1006
employee_id,1006
name,Arjun Das
department,Sales
salary,67000
experience,1
active,True


In [ ]:
df.loc['salary', 1]

KeyError: 1

### Whiteboard: `.loc` vs `.iloc`

This distinction trips up beginners consistently. Here is the clearest mental model:

```
.iloc  =  "integer location"  ->  always uses position numbers: 0, 1, 2...
.loc   =  "label"             ->  uses whatever the index actually IS
```

Right now our DataFrame's index IS 0, 1, 2... so they happen to give the same results. But once you filter or sort a DataFrame, the index labels and their positions diverge — and that is when the difference matters most. We will see this in a later session. For now: **use `.iloc` when you want "the 3rd row regardless of its label". Use `.loc` when you want "the row labelled X".**

In [ ]:
# Adding a new column — assign to a new column name
# Vectorized operations work exactly as in NumPy — no loop needed

# Salary per year of experience (a simple productivity proxy)
df["salary_per_exp_yr"] = df["salary"] / df["experience"]

# Seniority tier based on experience
df["seniority"] = pd.cut(
    df["experience"],
    bins   = [0, 3, 7, 100],
    labels = ["Junior", "Mid", "Senior"]
)

print(df[["name", "salary", "experience", "salary_per_exp_yr", "seniority"]])

---
### Exercise 2 — Working with a DataFrame

**Task:** Answer the questions using `df`.

In [ ]:
# Q1: Select only the 'name' and 'department' columns
q1 = df[['name', 'department']]

# Q2: Get the row at position 4 (fifth row) using iloc
q2 = df.iloc[4]

# Q3: Get Vikram's salary using .loc with the row label and column name
#     Vikram is at row label 3
q3 = df.loc[1004, 'salary']

# Q4: Add a new column called 'monthly_salary' = annual salary / 12
df['monthly_salary'] = df["salary"] / 12

# Q5: The average monthly salary (one line, using the new column)
q5 = df["monthly_salary"].mean()


print("Q1 name and department:")
print(q1)
print()
print("Q2 row at position 4:")
print(q2)
print()
print(f"Q3 Vikram's salary:    {q3}")
print(f"Q5 Avg monthly salary: {q5:,.0f}")
print()
print(f"Correct Q1 shape: {q1.shape == (6, 2)}")
print(f"Correct Q3: {q3 == 78000}")
print(f"Correct Q4 exists: {'monthly_salary' in df.columns}")

Q1 name and department:
                      name   department
employee_id                            
1001         Ananya Sharma  Engineering
1002           Rohit Mehta        Sales
1003            Priya Nair           HR
1004            Vikram Rao      Finance
1005          Sneha Pillai  Engineering
1006             Arjun Das        Sales

Q2 row at position 4:
employee_id            1005
name           Sneha Pillai
department      Engineering
salary               125000
experience                9
active                 True
Name: 1005, dtype: object

Q3 Vikram's salary:    78000
Q5 Avg monthly salary: 7,736

Correct Q1 shape: True
Correct Q3: True
Correct Q4 exists: True


<details>
<summary>Show solution</summary>

```python
q1 = df[["name", "department"]]
q2 = df.iloc[4]
q3 = df.loc[3, "salary"]
df["monthly_salary"] = df["salary"] / 12
q5 = df["monthly_salary"].mean()
```

</details>

---
# Part 4 — Reading a CSV File into a DataFrame

## The moment things get real

So far we have typed data by hand into Python dictionaries. In real work, data lives in files — usually CSV files. Pandas can load an entire file into a DataFrame in a single line.

Before we can read a file, we need one. The cell below creates a realistic employee CSV file — a company with 50 employees across four departments. It has deliberate messiness built in: missing values, mixed casing, and inconsistent formatting, because that is what real files look like.

In [ ]:
# Generate and save a realistic CSV file
# Students would normally receive this file from a data source or instructor
# We create it here so the notebook is self-contained

import csv, random
random.seed(42)

departments = ["Engineering", "Sales", "HR", "Finance"]
cities      = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Hyderabad"]
first_names = ["Ananya", "Rohit", "Priya", "Vikram", "Sneha", "Arjun",
               "Deepa", "Kiran", "Meera", "Suresh", "Lakshmi", "Rahul",
               "Divya", "Arun", "Pooja", "Sanjay", "Nisha", "Tarun"]
last_names  = ["Sharma", "Mehta", "Nair", "Rao", "Pillai", "Das",
               "Joshi", "Iyer", "Bose", "Kumar", "Reddy", "Singh"]

dept_salary_ranges = {
    "Engineering": (70000, 150000),
    "Sales":       (55000, 110000),
    "HR":          (50000,  95000),
    "Finance":     (65000, 130000),
}

rows = []
for i in range(50):
    emp_id  = 1001 + i
    name    = random.choice(first_names) + " " + random.choice(last_names)
    dept    = random.choice(departments)
    low, hi = dept_salary_ranges[dept]
    salary  = round(random.uniform(low, hi), -3)  # rounded to nearest 1000
    exp     = random.randint(1, 20)
    city    = random.choice(cities)
    perf    = round(random.uniform(2.0, 5.0), 1)  # performance rating
    year    = random.randint(2004, 2023)
    month   = random.randint(1, 12)
    join    = f"{year}-{month:02d}-01"
    active  = random.choice([True, True, True, False])  # 75% active

    # Introduce realistic messiness in ~15% of rows
    if random.random() < 0.08:
        salary = ""           # missing salary
    if random.random() < 0.08:
        perf = ""             # missing performance rating
    if random.random() < 0.06:
        dept = dept.lower()   # casing inconsistency

    rows.append([emp_id, name, dept, salary, exp, city, perf, join, active])

with open("employees.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["employee_id", "name", "department", "salary",
                     "experience", "city", "performance_rating", "join_date", "active"])
    writer.writerows(rows)

print("employees.csv created — 50 rows, 9 columns.")
print("Let's now load it with Pandas.")

employees.csv created — 50 rows, 9 columns.
Let's now load it with Pandas.


In [ ]:
# pd.read_csv() — read the file into a DataFrame in one line

emp = pd.read_csv("employees.csv")

print("File loaded successfully.")
print(f"Shape: {emp.shape}  ({emp.shape[0]} rows x {emp.shape[1]} columns)")

File loaded successfully.
Shape: (50, 9)  (50 rows x 9 columns)


In [ ]:
emp

,employee_id,name,department,salary,experience,city,performance_rating,join_date,active
0,1001,Vikram Sharma,HR,NaN,5,Mumbai,4.0,2021-02-01,False
1,1002,Ananya Bose,Sales,94000.0,18,Chennai,2.7,2022-05-01,True
2,1003,Meera Nair,Sales,108000.0,11,Mumbai,2.3,2007-06-01,True
3,1004,Tarun Mehta,Finance,70000.0,10,Hyderabad,NaN,2015-10-01,True
4,1005,Suresh Mehta,Sales,103000.0,13,Bangalore,3.4,2015-03-01,True
5,1006,Priya Kumar,Sales,84000.0,8,Delhi,3.4,2012-11-01,True
6,1007,Kiran Sharma,HR,68000.0,3,Delhi,4.7,2022-12-01,True
7,1008,Pooja Nair,HR,56000.0,18,Hyderabad,2.8,2022-07-01,False
8,1009,Sanjay Mehta,Engineering,139000.0,5,Delhi,4.4,2017-10-01,True
9,1010,Meera Bose,Engineering,124000.0,4,Hyderabad,4.3,2014-02-01,True


## `pd.read_csv()` — key parameters to know

`read_csv()` has dozens of optional parameters. These are the ones you will use most often:

| Parameter | What it does | Example |
|---|---|---|
| `filepath` | Path to the file (required) | `"employees.csv"` |
| `sep` | Column separator (default: `,`) | `sep=";"` for semicolon-delimited files |
| `header` | Which row is the header (default: 0) | `header=None` if there is no header row |
| `usecols` | Read only specific columns | `usecols=["name", "salary"]` |
| `nrows` | Read only the first N rows | `nrows=100` for large files |
| `skiprows` | Skip rows at the top | `skiprows=2` |
| `index_col` | Use a column as the row index | `index_col="employee_id"` |
| `parse_dates` | Parse a column as datetime | `parse_dates=["join_date"]` |
| `na_values` | Additional strings to treat as NaN | `na_values=["N/A", "-", "missing"]` |
| `encoding` | File text encoding | `encoding="utf-8"` |

In [ ]:
# Loading with commonly useful parameters

emp = pd.read_csv(
    "employees.csv",
    parse_dates  = ["join_date"],     # treat join_date as a datetime, not just a string
    index_col    = "employee_id"      # use employee_id as the row label instead of 0,1,2...
)

print(f"Shape: {emp.shape}")
print()
print("Column dtypes after loading with these parameters:")
print(emp.dtypes)
print()
print("Row index is now employee_id:")
print(emp.index[:5].tolist())

---
# Part 5 — Basic Inspection: Understanding a New Dataset

## The first thing you do with any new dataset

When you load a new dataset, you do not immediately start analysing it. You first **inspect** it — answer the basic questions: What columns are there? How many rows? What types? Are there missing values? Do the values look realistic?

Pandas gives you a standard set of tools for this. Every data scientist runs through roughly the same checklist every time they open a new file. Let's go through it.

In [ ]:
# 1. .head() and .tail() — see the first and last few rows
# The most important first step — a quick visual sanity check

print(".head() — first 5 rows:")
print(emp.head())
print()
print(".tail(3) — last 3 rows:")
print(emp.tail(3))

In [ ]:
# 2. Shape and column names

print(f".shape:    {emp.shape}   ({emp.shape[0]} rows, {emp.shape[1]} columns)")
print()
print(".columns:")
print(emp.columns.tolist())
print()
print(".index:")
print(f"  first few: {emp.index[:5].tolist()}")
print(f"  dtype: {emp.index.dtype}")

In [ ]:
# 3. .info() — the single most useful inspection command
# Shows: column names, non-null counts, and dtypes, all at once

print(".info():")
emp.info()

### Reading `.info()` output

Look at the `Non-Null Count` column. If any column shows fewer non-null values than the total row count, that column has **missing values**. For example, if `salary` shows `46 non-null` out of 50 rows, there are 4 missing salary values.

This is the fastest way to spot whether your data has gaps that need to be handled before analysis.

In [ ]:
# 4. .describe() — summary statistics for all numeric columns
# count, mean, std, min, 25th percentile, median, 75th percentile, max

print(".describe():")
print(emp.describe().round(1))

### Reading `.describe()` output

Each row tells you something different about each numeric column:

| Row | What to look for |
|---|---|
| `count` | Should match total rows — if it's lower, values are missing |
| `mean` | Typical value — is it reasonable? |
| `std` | Spread — is it surprisingly high or low? |
| `min` / `max` | Do the extremes make sense? A minimum salary of -5000 would be suspicious |
| `25%` / `50%` / `75%` | How spread out is the middle? Are there outliers pulling the mean away from the median? |

In [ ]:
# 5. Missing values — the most important quality check

# .isnull() returns a DataFrame of True/False — True where a value is missing
# .sum() adds them up per column
missing_counts = emp.isnull().sum()
print("Missing values per column:")
print(missing_counts)
print()

# Which columns have ANY missing values?
cols_with_missing = missing_counts[missing_counts > 0]
print("Columns with missing values:")
print(cols_with_missing)
print()

# What percentage of each column is missing?
missing_pct = (emp.isnull().sum() / len(emp) * 100).round(1)
print("Missing value percentage per column:")
print(missing_pct[missing_pct > 0])

In [ ]:
# 6. .value_counts() — frequency distribution of a categorical column
# Shows how many rows have each unique value

print("Department distribution:")
print(emp["department"].value_counts())
print()

print("City distribution:")
print(emp["city"].value_counts())
print()

# Notice: we earlier introduced a casing bug — 'engineering' vs 'Engineering'
# .value_counts() will surface this kind of inconsistency clearly
print("Does any department have casing issues?")
all_dept_values = emp["department"].value_counts()
print(all_dept_values)

In [ ]:
# 7. .nunique() — how many distinct values per column
# Useful to spot columns that should be categorical but have too many values
# (e.g. a 'country' column with 500 distinct values might have spelling errors)

print("Unique value counts per column:")
print(emp.nunique())
print()
print("employee_id should be 50 (all unique) — confirms no duplicate IDs")
print("department should be ~4 — if you see more, there may be casing/spelling issues")

In [ ]:
# 8. .sample() — view random rows instead of always the first or last
# Often surfaces issues that .head() misses because they appear later in the data

print("5 random rows (different every run unless you set random_state):")
print(emp.sample(5, random_state=1))

### The standard inspection checklist

When you open any new dataset, run through these in order:

```
1.  df.head()              — does it look like what I expected?
2.  df.shape               — how many rows and columns?
3.  df.info()              — what types? any missing values?
4.  df.describe()          — do the numbers make sense?
5.  df.isnull().sum()      — where exactly are the missing values?
6.  df['col'].value_counts() — what are the unique values in key categorical columns?
7.  df.nunique()           — are there unexpected numbers of unique values?
8.  df.sample(5)           — spot-check some random rows
```

You will use this checklist at the start of every data science task, for the rest of your career.

---
### Exercise 3 — Inspection

**Task:** Answer the questions below using the `emp` DataFrame and the inspection tools above. Do not look at the raw CSV — use only Pandas methods.

In [ ]:
# Q1: How many rows and columns does emp have?
rows, cols = emp.___
print(f"Q1: {rows} rows, {cols} columns")

# Q2: What is the mean salary? (round to 2 decimal places)
mean_salary = emp["salary"].___()
print(f"Q2: Mean salary = {mean_salary:,.2f}")

# Q3: How many employees are in each city?
city_counts = emp[___].value_counts()
print("Q3: Employees per city:")
print(city_counts)

# Q4: How many total missing values are there in the entire DataFrame?
total_missing = emp.___.sum().___()
print(f"Q4: Total missing values = {total_missing}")

# Q5: What are the unique cities in this dataset?
unique_cities = emp["city"].___().tolist()
print(f"Q5: Unique cities = {unique_cities}")

# Q6: Show the last 4 rows of the DataFrame
q6 = emp.___(4)
print("Q6: Last 4 rows:")
print(q6)

print()
print(f"Correct Q1: {rows == 50 and cols == 8}")
print(f"Correct Q4: {total_missing > 0}")
print(f"Correct Q5: {len(unique_cities) == 5}")

<details>
<summary>Show solution</summary>

```python
rows, cols    = emp.shape
mean_salary   = emp["salary"].mean()
city_counts   = emp["city"].value_counts()
total_missing = emp.isnull().sum().sum()
unique_cities = emp["city"].unique().tolist()
q6            = emp.tail(4)
```

</details>

---
## Final Exercise — End-to-End Challenge

You have just been handed a brand new employee dataset from a different company. Your job is to run the full inspection checklist and write a short summary of what you find.

Complete the cells below — each one runs one part of the checklist.

In [ ]:
# A second CSV file — different company, different structure, different problems

second_company_data = """staff_id,full_name,role,team,annual_ctc,tenure_months,location,rating
S001,Farhan Khan,Manager,Product,1800000,48,Pune,4.5
S002,Geetha Subramaniam,Analyst,Data,950000,12,Bangalore,3.8
S003,Himanshu Tiwari,Engineer,Platform,1200000,24,Delhi,
S004,Isha Malhotra,Designer,Product,880000,36,Mumbai,4.1
S005,Jagdish Patel,Analyst,Data,,18,Ahmedabad,3.2
S006,Kavitha Rajan,Manager,Platform,1550000,60,Chennai,4.7
S007,Lokesh Gupta,Engineer,data,1100000,9,Pune,3.9
S008,Madhuri Shetty,Designer,Product,790000,42,Mumbai,
S009,Nikhil Agarwal,Analyst,Data,920000,15,Delhi,4.0
S010,Omprakash Verma,Engineer,Platform,,30,Bangalore,3.6
S011,Preethi Nambiar,Manager,Data,1650000,72,Chennai,4.8
S012,Quaiser Siddiqui,Analyst,data,870000,8,Hyderabad,3.5
S013,Rashmi Desai,Engineer,Platform,1050000,20,Pune,4.2
S014,Sahil Mehra,Designer,product,760000,31,Mumbai,
S015,Tanuja Pillai,Manager,Product,1900000,84,Bangalore,4.9
"""

with open("second_company.csv", "w") as f:
    f.write(second_company_data)

print("second_company.csv created.")

In [ ]:
# Step 1: Load the file and peek at it

sc = pd.read_csv(___)
print("Shape:", sc.___)
print()
print(sc.___())

In [ ]:
# Step 2: Full info summary — types and non-null counts
sc.___()

In [ ]:
# Step 3: Numeric summary statistics
print(sc.___().round(1))

In [ ]:
# Step 4: Missing values — which columns and how many
print("Missing values per column:")
print(sc.___.sum())
print()
print(f"Total missing: {sc.isnull().sum().___()}")

In [ ]:
# Step 5: Spot the data quality issue in the 'team' column
print("Team value counts:")
print(sc[___].value_counts())
print()
print("Unique team values:", sc["team"].___(). tolist())
print()
print("What problem do you see?")

In [ ]:
# Step 6: Answer these questions using only Pandas methods

# A: How many unique roles are there?
num_roles = sc["role"].___()

# B: Who has the highest annual_ctc? (use .idxmax() — gives the INDEX of the max value)
highest_ctc_idx = sc["annual_ctc"].___()
highest_ctc_name = sc.loc[highest_ctc_idx, "full_name"]

# C: Average tenure in months
avg_tenure = sc[___].mean()


print(f"A: Unique roles:              {num_roles}")
print(f"B: Highest earner:            {highest_ctc_name}")
print(f"C: Average tenure (months):   {avg_tenure:.1f}")
print()
print(f"Correct A (3 roles):      {num_roles == 3}")
print(f"Correct B (Tanuja):       {'Tanuja' in highest_ctc_name}")
print(f"Correct C (> 20 months):  {avg_tenure > 20}")

<details>
<summary>Show solution</summary>

```python
# Step 1
sc = pd.read_csv("second_company.csv")
print("Shape:", sc.shape)
sc.head()

# Step 2
sc.info()

# Step 3
sc.describe()

# Step 4
sc.isnull().sum()
sc.isnull().sum().sum()

# Step 5
sc["team"].value_counts()
sc["team"].unique().tolist()

# Step 6
num_roles        = sc["role"].nunique()
highest_ctc_idx  = sc["annual_ctc"].idxmax()
avg_tenure       = sc["tenure_months"].mean()
```

**Data quality issue in Step 5:** The `team` column has inconsistent casing — `"data"`, `"product"`, and `"platform"` (lowercase) appear alongside `"Data"`, `"Product"`, and `"Platform"` (title case). This means the same team is being counted as two separate values. Fixing this (`.str.title()` or `.str.lower()`) is covered in the next session on data cleaning.

</details>

---
## Session Summary

### Why Pandas?
NumPy can only hold one data type per array. Real datasets mix integers, floats, strings, booleans, and dates in the same table. Pandas solves this with the DataFrame — a labelled table where each column can hold a different type. It is built on NumPy, so all the vectorized speed carries over.

### Series
- A one-dimensional labelled array — one column of a DataFrame
- Has an **index** (row labels) and a **name** (the column name)
- Supports all the same vectorized operations as NumPy (masking, arithmetic, aggregation)
- Labels travel with values — sorting or filtering never breaks the association

### DataFrame
- A two-dimensional labelled table — a dictionary of Series sharing the same index
- Access columns: `df['col']` (bracket, always works) or `df.col` (dot, only for simple names)
- Access multiple columns: `df[['col1', 'col2']]` (list inside brackets)
- Access rows: `.iloc[n]` by position, `.loc[label]` by label
- Access a cell: `.loc[row, col]` or `.iloc[row_pos, col_pos]`
- Add a column: `df['new_col'] = expression`

### Reading CSV
- `pd.read_csv("file.csv")` loads a file into a DataFrame in one line
- `parse_dates=["col"]` — tell Pandas to parse that column as datetime
- `index_col="col"` — use a column as the row index
- `usecols=[...]` — read only specific columns
- `nrows=n` — read only the first n rows (useful for large files)

### The inspection checklist

```python
df.head()               # first 5 rows
df.tail()               # last 5 rows
df.shape                # (rows, columns)
df.columns.tolist()     # column names
df.info()               # types + non-null counts
df.describe()           # numeric summary statistics
df.isnull().sum()       # missing values per column
df['col'].value_counts()  # frequency of each value in a column
df.nunique()            # distinct values per column
df.sample(5)            # random rows
```

---

### What comes next

You can now load a real dataset and understand its shape, types, and quality at a glance. The next session covers **data cleaning and filtering** — using the problems you just found (missing values, casing inconsistencies, outliers) and fixing them systematically with Pandas.

---

### Quick reference

```python
import pandas as pd

# Series
s = pd.Series(data, index=labels, name="col")
s["label"]          # access by label
s.iloc[2]           # access by position
s[s > 90000]        # boolean filter
s.mean()  s.max()  s.sort_values()

# DataFrame creation
df = pd.DataFrame({"col1": [...], "col2": [...]})

# Reading
df = pd.read_csv("file.csv", parse_dates=["date_col"], index_col="id_col")

# Column access
df["salary"]                  # one column -> Series
df[["name", "salary"]]        # multiple columns -> DataFrame

# Row access
df.iloc[0]                    # row by position
df.loc[label]                 # row by label
df.loc[row, "col"]            # single cell

# Inspection
df.head()  df.tail()  df.shape  df.info()  df.describe()
df.isnull().sum()  df["col"].value_counts()  df.nunique()
```